In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
df1_path = "../../data/dataset1/dataset1.csv"
df1_weather_path = "../../data/weather/dataset1_weather.csv"

df1_raw = pd.read_csv(df1_path)
weather_raw = pd.read_csv(df1_weather_path)

print('Dataset 1 raw shape:', df1_raw.shape)
print('Weather raw shape:', weather_raw.shape)

display(df1_raw.head())
display(weather_raw.head())

In [ ]:
df1 = df1_raw.copy()

# Giữ lại weather gốc của Dataset 1 để đối chiếu sau khi join Open-Meteo.
df1 = df1.rename(
    columns={
        'weather_condition': 'original_weather_condition',
        'temperature_c': 'original_temperature_c',
    }
)

df1['timestamp_datetime'] = pd.to_datetime(df1['timestamp'], errors='coerce')
df1['date'] = df1['timestamp_datetime'].dt.normalize()
df1['hour'] = df1['timestamp_datetime'].dt.hour
df1['location'] = df1['city']

print('Invalid timestamp_datetime:', df1['timestamp_datetime'].isna().sum())
print('Dataset 1 date range:', df1['date'].min(), '->', df1['date'].max())
print('Dataset 1 locations:')
display(df1['location'].value_counts())

display(
    df1[
        [
            'timestamp',
            'timestamp_datetime',
            'date',
            'hour',
            'location',
            'original_temperature_c',
            'original_weather_condition',
            'total_amount',
        ]
    ].head()
)

In [ ]:
weather = weather_raw.copy()
weather['date'] = pd.to_datetime(weather['date'], errors='coerce').dt.normalize()
weather['hour'] = pd.to_numeric(weather['hour'], errors='coerce').astype('Int64')

weather_cols = [
    'date',
    'hour',
    'location',
    'city',
    'borough',
    'temperature_c',
    'apparent_temperature_c',
    'relative_humidity',
    'precipitation_mm',
    'rain_mm',
    'weather_code',
    'weather_code_description',
    'weather_condition',
    'wind_speed_10m_kmh',
    'cloud_cover_pct',
    'is_rainy',
    'is_snowy',
    'is_precipitation',
    'weather_latitude',
    'weather_longitude',
    'weather_source',
    'weather_spatial_level',
]

weather = weather[weather_cols].copy()

print('Weather duplicate join keys:', weather.duplicated(['date', 'hour', 'location']).sum())
print('Weather date range:', weather['date'].min(), '->', weather['date'].max())
print('Weather locations:')
display(weather['location'].value_counts())

display(weather.head())

In [ ]:
dataset1_locations = set(df1['location'].dropna().unique())
weather_locations = set(weather['location'].dropna().unique())

print('Locations in Dataset 1 but not in weather:', sorted(dataset1_locations - weather_locations))
print('Locations in weather but not in Dataset 1:', sorted(weather_locations - dataset1_locations))

df1_join_keys = df1[['date', 'hour', 'location']].drop_duplicates()
weather_join_keys = weather[['date', 'hour', 'location']].drop_duplicates()

key_check = df1_join_keys.merge(
    weather_join_keys.assign(has_weather_key=True),
    on=['date', 'hour', 'location'],
    how='left',
)

print('Dataset 1 unique join keys:', len(df1_join_keys))
print('Join keys without weather:', key_check['has_weather_key'].isna().sum())

display(key_check[key_check['has_weather_key'].isna()].head(20))

In [ ]:
df1_with_weather = df1.merge(
    weather,
    on=['date', 'hour', 'location'],
    how='left',
    validate='many_to_one',
)

print('Dataset 1 before join:', df1.shape)
print('Dataset 1 after join:', df1_with_weather.shape)

display(df1_with_weather.head())

In [ ]:
weather_check_cols = [
    'temperature_c',
    'apparent_temperature_c',
    'relative_humidity',
    'precipitation_mm',
    'rain_mm',
    'weather_code',
    'weather_condition',
    'wind_speed_10m_kmh',
    'cloud_cover_pct',
]

missing_weather = df1_with_weather[weather_check_cols].isna().sum().sort_values(ascending=False)
missing_weather_pct = (df1_with_weather[weather_check_cols].isna().mean() * 100).sort_values(ascending=False)

weather_missing_summary = pd.DataFrame({
    'missing_count': missing_weather,
    'missing_pct': missing_weather_pct,
})

print('Missing weather after join:')
display(weather_missing_summary)

print('Weather condition distribution:')
display(df1_with_weather['weather_condition'].value_counts(dropna=False))

print('Temperature summary by location:')
display(df1_with_weather.groupby('location')['temperature_c'].describe())

print('Comparison with original Dataset 1 weather fields:')
display(
    df1_with_weather[
        [
            'original_temperature_c',
            'temperature_c',
            'original_weather_condition',
            'weather_condition',
        ]
    ].head(20)
)

In [ ]:
OUTPUT_PATH = Path('../../data/dataset1/dataset1_with_weather.csv')

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df1_with_weather.to_csv(OUTPUT_PATH, index=False, encoding='utf-8')

print('Saved Dataset 1 with weather to:', OUTPUT_PATH)
print('Final shape:', df1_with_weather.shape)